# Missingness EDA (Team 09, Anish)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the repo root (works whether the kernel starts at the repo root or in notebooks/)
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
forecast_path = root / "data" / "raw" / "AA_PAX_FORECASTING.csv"
airport_path = root / "data" / "raw" / "AA_AIRPORT_INFO.csv"

df = pd.read_csv(forecast_path)          # forecasting file
airports = pd.read_csv(airport_path)     # airport lookup file

EXPECTED_ROWS = 1_533_522
n_rows = len(df)
assert n_rows == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS:,} rows, loaded {n_rows:,}"

print(f"AA_PAX_FORECASTING: {n_rows:,} rows, {df.shape[1]} columns")
print(f"AA_AIRPORT_INFO: {len(airports):,} rows, {airports.shape[1]} columns")

## Missing values

Percent missing for every column in both files, computed on the full files.

In [ ]:
def missing_table(d):
    return (
        pd.DataFrame({
            "dtype": d.dtypes.astype(str),
            "n_missing": d.isna().sum(),
            "pct_missing": (d.isna().mean() * 100).round(2),
        })
        .sort_values("pct_missing", ascending=False)
    )

print(f"Forecasting file: {n_rows:,} rows")
display(missing_table(df))

print(f"Airport file: {len(airports):,} rows")
display(missing_table(airports))

The forecasting file has 0% missing in every column, and the placeholder checks below confirm that no gaps are hidden as text or sentinel values. A numeric 0 is not a missing value here.

In [ ]:
placeholders = {"", "?", "-", "--", "unknown", "n/a", "na", "nan", "null", "none", "missing", "tbd"}
for c in ["ORG_AIRPRT", "DES_AIRPRT", "SF_LOAD_TMS"]:
    s = df[c].astype(str).str.strip().str.lower()
    print(f"{c}: {df[c].nunique():,} unique, {s.isin(placeholders).sum():,} placeholder-like values")

for c in ["OD_PAX", "OD_REV_USD"]:
    print(f"{c}: zeros = {(df[c] == 0).sum():,}, negatives = {(df[c] < 0).sum():,}")

print("Duplicate rows:", df.duplicated().sum())
print("Constant columns:", [c for c in df.columns if df[c].nunique() == 1])

Pandas reads the text `NA` as a missing value by default, and `NA` is also the country code for Namibia. The next cell re-reads the airport file without that conversion to see whether any of the blank country or state values are really the text `NA`.

In [ ]:
raw_airports = pd.read_csv(airport_path, keep_default_na=False)
for c in ["CNTRY_CD", "STATE_PROVNC_CD"]:
    print(f"{c}: literal 'NA' = {(raw_airports[c] == 'NA').sum():,} | empty strings = {(raw_airports[c] == '').sum():,}")

## Where the gaps are

The forecasting file has no NaN values, so there is nothing to locate by year, month, origin or destination there. The gaps are in the airport file.

In [ ]:
print("NaN cells in forecasting file:", int(df.isna().sum().sum()))

state_missing = airports["STATE_PROVNC_CD"].isna()
country = airports["CNTRY_CD"]

# group airports by country: US, non-US, or country itself missing
group = np.where(country.isna(), "country missing", np.where(country == "US", "US", "non-US"))
split = pd.crosstab(pd.Series(group, name="country group"),
                    state_missing.map({True: "state missing", False: "state present"}).rename("state"))
display(split)

print(f"Airports with STATE_PROVNC_CD missing: {state_missing.sum():,} of {len(airports):,} ({state_missing.mean() * 100:.1f}%)")
print(f"Airports with CNTRY_CD missing: {country.isna().sum():,}; of those, state also missing: {(country.isna() & state_missing).sum():,}")

In [ ]:
# Check if the airport gaps land on the origin or destination of forecasting rows
# Find the airport-code column in the airport file: the one whose values match the forecasting airport codes
codes = set(df["ORG_AIRPRT"].unique()) | set(df["DES_AIRPRT"].unique())
matches = {c: int(airports[c].isin(codes).sum()) for c in airports.columns}
key_col = max(matches, key=matches.get)
print(f"Airport code column: {key_col} | rows matching forecasting codes: {matches[key_col]:,}")
print(f"Duplicate values in {key_col}:", airports[key_col].duplicated().sum())

lookup = airports.drop_duplicates(key_col).set_index(key_col)

for col, label in [("ORG_AIRPRT", "origin"), ("DES_AIRPRT", "destination")]:
    state_gap = df[col].map(lookup["STATE_PROVNC_CD"].isna())     # NaN if code not in lookup
    print(f"{label}: codes not found in airport file = {state_gap.isna().sum():,} rows | "
          f"state missing for {state_gap.eq(True).mean() * 100:.1f}% of rows")

In [ ]:
# Check entirely missing rows
print(pd.crosstab(df["DEP_YEAR"], df["DEP_MONTH"]).to_string())

## Decisions

| Column | % missing | Decision | Why |
|---|---|---|---|
| `STATE_PROVNC_CD` (airport file) | about half of airports | **keep** | The blanks are all non-US airports, where a state or province code does not apply, so imputing would invent values. |
| `CNTRY_CD` (airport file) | 8 rows | **keep** | Only 8 rows, and all 8 are also missing a state, so I label them as unknown instead of guessing a country. |
| Forecasting file, all 8 columns | 0% | **keep** (except two) | Nothing is missing. Drop `Unnamed: 0` (a leftover row index) and `SF_LOAD_TMS` (the same timestamp in every row). |

The missing airport values follow a pattern (non-US airports) and are not random. The forecasting file is complete in every cell, and its only incompleteness is rows that do not exist: 2026 stops partway through the year and 2020 is lower.

## Plots

Plots use a 50,000-row sample (`random_state=9`) of the forecasting file. The numbers in the summary come from the full file.

In [ ]:
sample = df.sample(n=50_000, random_state=9)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(np.log10(sample["OD_PAX"]), bins=50)
axes[0].set_title("log10(OD_PAX), 50k sample")
axes[0].set_xlabel("log10(passengers)")

axes[1].hist(np.log10(sample["OD_REV_USD"].clip(lower=1)), bins=50)
axes[1].set_title("log10(OD_REV_USD), 50k sample")
axes[1].set_xlabel("log10(revenue USD)")

sample["DEP_YEAR"].value_counts().sort_index().plot.bar(ax=axes[2])
axes[2].set_title("Rows per DEP_YEAR, 50k sample")
axes[2].set_xlabel("DEP_YEAR")

plt.tight_layout()
plt.show()

In [ ]:
# Full-file summary statistics that back the distributions paragraph
for c in ["OD_PAX", "OD_REV_USD"]:
    s = df[c]
    print(f"{c}: mean={s.mean():,.1f}  median={s.median():,.1f}  "
          f"mean/median={s.mean() / s.median():.1f}  skew={s.skew():.1f}  max={s.max():,.1f}")

## Summary

**Shape.** The forecasting file has 1,533,522 rows and 8 columns, with no duplicate rows. Each row is one route (original airport to destination airport) in one month. It covers 150 origin and 150 destination airports from 2019 to 2026. Some periods have fewer rows. The full years have about 205,000 rows, 2020 has fewer, and 2026 stops partway through the year. The airport file is a lookup table that joins to the forecasting file on the airport codes. Two of the eight forecasting columns are not useful. `Unnamed: 0` is just a row number, and `SF_LOAD_TMS` has the same timestamp in every row.

**Distributions.** `OD_PAX` and `OD_REV_USD` are both skewed to the right: a few very big routes have huge values, but most routes are small. For `OD_PAX`, the average is about 2,765 but the middle value is only about 503, and the largest is near 140,000. For `OD_REV_USD`, the average is about $688K, the middle value is about $156K, and the largest is near $121M. Because of this, plots should use a log scale, and a log transform may help linear models. `OD_PAX` also has decimals (the smallest is 0.13), so the counts are probably estimates and not exact counts.

**Leakage.** The target is `OD_PAX`. `OD_REV_USD` is the main risk, because it is the money earned from those same passengers, and we only know it after they have flown. It should not be used as a feature, or only its past values should be used. `Unnamed: 0` and `SF_LOAD_TMS` should be dropped. Validation should split by time (train on earlier months, test on later ones), because a random split would let the model train on months that come after the ones it is tested on.

**Model.** Gradient-boosted trees fit well here. The missing values are structural: `STATE_PROVNC_CD` is blank for airports outside the US. Trees like LightGBM, XGBoost, and scikit-learn's HistGradientBoosting can use blank values directly, so we do not need to make up states or fill anything in.